# Generate heterostructures

Builds coherent film/substrate interfaces directly from **two already-cut slab files**
(e.g. the ones `generate-slabs.ipynb` produces; give them already relaxed, or let the ML-IAP relax them, see below) -- no bulk
unit cell needed. For a given substrate slab and film slab:

1. `ZSLGenerator` is run directly on the two slabs' in-plane lattice vectors to find
   coincidence (coherent) matches. This is purely linear algebra on the two 2-D lattices --
   it never looks at crystal symmetry, so it treats a cubic/hexagonal pair exactly like an
   orthorhombic/trigonal pair. Lower-symmetry or more mismatched pairs just tend to need a
   larger `max_interface_area` (bigger supercells) or looser tolerances before any match is
   found at all -- check the printed match counts below for your specific pair.
2. Each match pairs particular supercell vectors of the film with particular supercell vectors of the substrate, which
   fixes the **relative in-plane orientation (twist)** of the two crystals. That *effective twist* is computed for every match
   and stored with the candidate. Pre-rotating the film before the matching does **not** explore twist angles: ZSL only uses
   lengths and angles of the lattice vectors, so a rigid rotation returns exactly the same matches, and the interface is built
   on the substrate lattice, so the rotation would be lost anyway.
3. For each match, symmetry-distinct **in-plane registries** (lateral stacking offsets) are enumerated with
   `Interface.get_shifts_based_on_adsorbate_sites`, and a small grid of **gaps** is applied.

All of this produces a *controlled, finite* set of inequivalent candidate interface `POSCAR` files, with one parameter per axis
(`n_matches`, `n_registries_per_match`, `n_gaps`) and a single overall cap (`max_total_candidates`) if their product is too large.

Relaxing/ranking those candidates with the ML-IAP is a **separate, optional** step (`run_mliap_relaxation`), off by default:
every candidate `POSCAR` is left untouched (relaxation results go into the `relaxation/` folder of each candidate), so the generated set
can be handed straight to VASP/DFT instead.

**On substrate vs. film:** you do have to designate one slab as substrate and one as film -- it determines the geometry
(substrate stays put, film is flipped and stacked on top with the given gap/vacuum) -- but the match-finding itself
(`bidirectional=True` below) searches supercells starting from either lattice.

**On the film being strained:** the interface is built on the substrate in-plane lattice, so the film is strained to it. The
film and substrate reference energies in the adhesion energy are therefore evaluated **in that same strained cell**, so the
adhesion energy does not contain the strain energy of the film.

**On checking the surface:** a Miller index is only meaningful relative to a bulk cell, so there is no way to verify from a bare
slab file that it truly *is* the surface its name claims. We only check that the file *looks* like a slab (a large vacuum gap
along one lattice vector) -- see `heterostructure.check_slab_file`.

In [1]:
import json
import os
import shutil

from importlib.metadata import version
from operator import itemgetter

import numpy as np
import matplotlib.pyplot as plt

from pymatgen.core.structure import Structure

from libraries import energy_lookup, heterostructure, vasp_files


/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


## Inputs -- two already-cut slabs, plus the Miller index each was cut along

In [2]:
general_folder = 'input/heterostructure'

substrate_miller = (0, 0, 1)
substrate_POSCAR = "input/test/slab_v1/0_0_1_i_8/POSCAR"

film_miller = (1, 1, 1)
film_POSCAR = "input/test/slab_v1/1_1_1_i_24/POSCAR"


substrate_structure  = Structure.from_file(substrate_POSCAR)
film_structure       = Structure.from_file(film_POSCAR)
heterostructure.check_slab_file(substrate_structure, 'substrate', substrate_POSCAR, expected_miller=substrate_miller)
heterostructure.check_slab_file(film_structure,      'film',      film_POSCAR,      expected_miller=film_miller)

substrate_slab = heterostructure.wrap_as_slab(substrate_structure, substrate_miller)
film_slab      = heterostructure.wrap_as_slab(film_structure, film_miller)


substrate: in-plane area 37.3 Å², vacuum along the normal 22.4 Å
film: in-plane area 112.0 Å², vacuum along the normal 21.9 Å


## Configuration

In [3]:
os.makedirs(general_folder, exist_ok=True)

vacuum_over_film = 20.0   # Å, vacuum left over the top of the film

# Search space for candidate generation -- one clear parameter per axis
gap_bounds             = (1.5, 4.0)   # interlayer gap, Å
n_gaps                 = 3            # how many gaps to sample in gap_bounds (1 = single fixed gap)
n_matches              = 10           # ZSL matches kept, ranked by lowest strain first (each has its own effective twist)
n_registries_per_match = 4            # symmetry-distinct in-plane registries kept per match

# Single overall cap: if matches × registries × gaps exceeds this, the candidate list is
# subsampled evenly down to this many -- the one number to turn if you just want
# "give me roughly N heterostructures to look at".
max_total_candidates = 60

# ZSL matching tolerances -- the main lever for "different lattice and symmetry": for a
# substrate/film pair with very mismatched lattices or low symmetry you may need a larger
# max_interface_area or looser tolerances before any coherent match exists at all
max_interface_area  = 400.0  # Å², cap on the coincidence supercell area
max_area_ratio_tol  = 0.1
max_length_tol      = 0.1
max_angle_tol       = 0.1

# Whether to relax/rank the generated candidates with the ML-IAP. Leave this off to just get
# the candidate POSCARs (e.g. to hand them to VASP/DFT instead). Every POSCAR is left
# untouched either way -- relaxation results go into the relaxation/ folder of each candidate.
run_mliap_relaxation = True
model_load_path      = 'mace-mpa-0-medium.model'
device               = 'cpu'    # switch to 'cuda' if a GPU is available


heterostructure_data = {
    'substrate_miller':       substrate_miller,
    'film_miller':            film_miller,
    'vacuum_over_film':       vacuum_over_film,
    'gap_bounds':             gap_bounds,
    'n_gaps':                 n_gaps,
    'n_matches':              n_matches,
    'n_registries_per_match': n_registries_per_match,
    'max_total_candidates':   max_total_candidates,
    'max_interface_area':     max_interface_area,
    'max_area_ratio_tol':     max_area_ratio_tol,
    'max_length_tol':         max_length_tol,
    'max_angle_tol':          max_angle_tol,
    'run_mliap_relaxation':   run_mliap_relaxation,
    'model_load_path':        model_load_path,
    'device':                 device,
}

with open(f'{general_folder}/heterostructure_data.json', 'w') as json_file:
    json.dump(heterostructure_data, json_file, indent=2)

# Copy the original slab files there, untouched, for provenance
shutil.copy(substrate_POSCAR, f'{general_folder}/POSCAR-substrate')
shutil.copy(film_POSCAR,      f'{general_folder}/POSCAR-film')


'input/heterostructure/POSCAR-film'

## Adhesion energy (used only if `run_mliap_relaxation=True`)

\begin{equation}
    \Delta G_{\text{adhesion}} = \frac{G_{\text{interface}} - G_{\text{film}} - G_{\text{substrate}}}{A}
\end{equation}

analogous to `surfaces.get_surface_energy_of_formation`, but normalized by a single interface (not the two free surfaces of an
isolated slab). $G$ is the free (Gibbs) energy; here it is evaluated at the **0 K** level, $G = E_0$, the total energy of the relaxed
structure, so the quantity printed and stored as `adhesion_energy` is the 0 K value. The last section of the notebook (optional)
evaluates it at a temperature, $G(T) = U_0(T) + F_{vib}(T)$, from NPT runs. $G_{\text{film}}$ and $G_{\text{substrate}}$ are the free
energies of the film and the substrate atoms alone, in the
**same strained interface cell** and relaxed at fixed cell, so the strain energy of the film is not counted as adhesion. $A$ is the interface area and the result is converted to J/m² with the same eV/Å²
factor used elsewhere in this repo.

The formula is implemented in `heterostructure.get_adhesion_energy_of_formation` (`libraries/heterostructure.py`).

## Generate the candidate interfaces

The in-plane registries are fractional shifts of the film. `Interface.from_slabs` only stores them and never moves the film, so `heterostructure.build_interface` applies them explicitly; registries that differ by a lattice translation are merged.


In [4]:
gap_grid = np.linspace(*gap_bounds, n_gaps) if n_gaps > 1 else [float(np.mean(gap_bounds))]

print('pymatgen', version('pymatgen'))
print('Substrate in-plane lattice:', heterostructure.describe_lattice(substrate_slab))
print('Film in-plane lattice:     ', heterostructure.describe_lattice(film_slab))

zsl_generator = heterostructure.make_zsl_generator(max_interface_area, max_area_ratio_tol, max_length_tol, max_angle_tol)
matches = heterostructure.find_matches(film_slab, substrate_slab, zsl_generator, n_matches)
print(f'{len(matches)} coherent match(es) with max_area={max_interface_area:g} Å², length tol={max_length_tol:.0%}, angle tol={max_angle_tol:.0%}')

if not matches:
    heterostructure.explain_no_match(film_slab, substrate_slab, max_interface_area, max_area_ratio_tol)
    raise RuntimeError('No coherent match between these two slabs with the configured settings (see the lattices and the report above).')

candidates = []  # metadata only -- structures are (re)built once we know which ones survive the cap
for m_idx, match in enumerate(matches):
    twist = heterostructure.effective_twist(match, film_slab, substrate_slab)
    film_sl_slab, sub_sl_slab = heterostructure.build_supercell_slabs(film_slab, substrate_slab, match)
    registries = heterostructure.enumerate_registries(film_sl_slab, sub_sl_slab, n_registries_per_match,
                                          reference_gap=float(np.mean(gap_bounds)), vacuum_over_film=vacuum_over_film)
    print(f'match {m_idx}: strain={heterostructure.match_strain(match):.4f}, effective twist={twist:.1f}°, '
          f'area={match.match_area:.1f} Å², {len(registries)} registries')
    for r_idx, registry in enumerate(registries):
        for g_idx, gap in enumerate(gap_grid):
            candidates.append({
                'twist': twist,
                'match_idx': m_idx, 'strain': float(heterostructure.match_strain(match)),
                'registry_idx': r_idx, 'registry': registry,
                'gap_idx': g_idx, 'gap': float(gap),
                'film_sl_slab': film_sl_slab, 'sub_sl_slab': sub_sl_slab,
            })

print(f'{len(candidates)} candidate interface(s) before capping')

if len(candidates) > max_total_candidates:
    candidates = heterostructure.subsample_evenly(candidates, max_total_candidates)
    print(f'subsampled down to max_total_candidates={max_total_candidates}')

print(f'{len(candidates)} candidate interface(s) to write')

pymatgen 2026.5.4
Substrate in-plane lattice: |a| = 4.24 Å, |b| = 8.80 Å, angle = 90.0°, area = 37.3 Å²
Film in-plane lattice:      |a| = 9.77 Å, |b| = 11.61 Å, angle = 80.9°, area = 112.0 Å²
10 coherent match(es) with max_area=400 Å², length tol=10%, angle tol=10%
match 0: strain=0.0211, effective twist=nan°, area=336.0 Å², 4 registries
match 1: strain=0.0211, effective twist=114.7°, area=336.0 Å², 4 registries
match 2: strain=0.0368, effective twist=nan°, area=336.0 Å², 4 registries
match 3: strain=0.0368, effective twist=-34.7°, area=336.0 Å², 4 registries
match 4: strain=0.0496, effective twist=98.8°, area=336.0 Å², 4 registries
match 5: strain=0.0496, effective twist=nan°, area=336.0 Å², 4 registries
match 6: strain=0.0610, effective twist=118.7°, area=224.0 Å², 4 registries
match 7: strain=0.0610, effective twist=nan°, area=224.0 Å², 4 registries
match 8: strain=0.0774, effective twist=31.7°, area=336.0 Å², 4 registries
match 9: strain=0.0774, effective twist=nan°, area=336.0 Å²,

In [5]:
for c_idx, candidate in enumerate(candidates):
    candidate_folder = (
        f'{general_folder}/match_{candidate["match_idx"]}/'
        f'registry_{candidate["registry_idx"]}/gap_{candidate["gap_idx"]}'
    )
    reference_folder = f'{general_folder}/match_{candidate["match_idx"]}/reference'

    film_sl_slab = candidate['film_sl_slab']
    sub_sl_slab  = candidate['sub_sl_slab']

    # The registry (lateral shift of the film) is applied explicitly: Interface.from_slabs ignores in_plane_offset
    interface = heterostructure.build_interface(film_sl_slab, sub_sl_slab, candidate['registry'], candidate['gap'], vacuum_over_film)
    interface_area = np.linalg.norm(np.cross(interface.lattice.matrix[0], interface.lattice.matrix[1]))

    heterostructure.write_structure(interface, f'{candidate_folder}/POSCAR')
    vasp_files.prepare_relaxation_folder(candidate_folder)   # relaxation/ with a copy of the inputs

    # Isolated film/substrate references: the film and substrate atoms of the interface, alone, in the SAME
    # (strained) interface cell. They only depend on the match, not on registry/gap, so they are written once per
    # match and shared -- energy_lookup.read_energy only relaxes them (at fixed cell) the first time it sees the folder.
    if not os.path.exists(f'{reference_folder}/film/POSCAR'):
        film_reference, substrate_reference = heterostructure.split_interface(interface)
        heterostructure.write_structure(film_reference,      f'{reference_folder}/film/POSCAR')
        heterostructure.write_structure(substrate_reference, f'{reference_folder}/substrate/POSCAR')
        vasp_files.prepare_relaxation_folder(f'{reference_folder}/film')
        vasp_files.prepare_relaxation_folder(f'{reference_folder}/substrate')

    with open(f'{candidate_folder}/candidate_data.json', 'w') as json_file:
        json.dump({
            'substrate_miller':  substrate_miller,
            'film_miller':       film_miller,
            'effective_twist':   candidate['twist'],
            'strain':            candidate['strain'],
            'registry':          [float(x) for x in candidate['registry']],
            'gap':               candidate['gap'],
            'interface_area':    float(interface_area),
            'reference_folder':  reference_folder,
        }, json_file, indent=2)

    candidates[c_idx]['candidate_folder'] = candidate_folder
    candidates[c_idx]['reference_folder'] = reference_folder
    candidates[c_idx]['interface_area']   = float(interface_area)

print(f'Wrote {len(candidates)} candidate interface POSCARs under {general_folder}/')

Wrote 60 candidate interface POSCARs under input/heterostructure/


## Optional: relax and rank with the ML-IAP

Only runs if `run_mliap_relaxation = True` above. Every candidate `POSCAR` written above is
left as-is; this reads/writes the `CONTCAR` and energy files in the `relaxation/` folder of each candidate via the existing
`energy_lookup.read_energy`/`libraries.mlip` pipeline, exactly like `iterate-slabs.ipynb` and
`H-absorption.ipynb` do. Relaxations keep the cell fixed (coherent interface).

In [ ]:
results = []
if run_mliap_relaxation:
    # Interface, film and substrate energies must come from the same level of theory
    energy_lookup.check_energy_sources(
        [c['candidate_folder'] for c in candidates]
        + [f'{c["reference_folder"]}/{part}' for c in candidates for part in ('film', 'substrate')],
        label='adhesion energy', model_load_path=model_load_path)

    for candidate in candidates:
        film_energy      = energy_lookup.read_energy(f'{candidate["reference_folder"]}/film',      model_load_path=model_load_path, device=device)
        substrate_energy = energy_lookup.read_energy(f'{candidate["reference_folder"]}/substrate', model_load_path=model_load_path, device=device)
        interface_energy = energy_lookup.read_energy(candidate['candidate_folder'],                  model_load_path=model_load_path, device=device)

        if any(e is None or np.isnan(e) for e in (film_energy, substrate_energy, interface_energy)):
            print(f'{candidate["candidate_folder"]}: relaxation/energy failed, skipping')
            continue

        adhesion_energy = heterostructure.get_adhesion_energy_of_formation(
            interface_energy, film_energy, substrate_energy, candidate['interface_area']
        )

        with open(f'{candidate["candidate_folder"]}/candidate_data.json') as json_file:
            candidate_data = json.load(json_file)
        candidate_data.update({
            'film_energy':      float(film_energy),
            'substrate_energy': float(substrate_energy),
            'interface_energy': float(interface_energy),
            'adhesion_energy':  float(adhesion_energy),
        })
        with open(f'{candidate["candidate_folder"]}/candidate_data.json', 'w') as json_file:
            json.dump(candidate_data, json_file, indent=2)

        results.append({'folder': candidate['candidate_folder'], 'adhesion_energy': float(adhesion_energy),
                        'effective_twist': candidate['twist']})
        print(f'{candidate["candidate_folder"]}: E_adhesion = {adhesion_energy:.4g} J/m²')

    results.sort(key=itemgetter('adhesion_energy'))
    with open(f'{general_folder}/heterostructure_results.json', 'w') as json_file:
        json.dump(results, json_file, indent=2)

    if results:
        print(f'\nBest: {results[0]["folder"]}  E_adhesion = {results[0]["adhesion_energy"]:.4g} J/m²')
else:
    print('run_mliap_relaxation is False -- candidate POSCARs were generated but not relaxed/ranked.')

Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


      Step     Time          Energy          fmax
BFGS:    0 18:12:41     -749.064960        1.572403
BFGS:    1 18:12:42     -750.085257        1.426338
BFGS:    2 18:12:43     -754.137718        1.931811
BFGS:    3 18:12:44     -754.638213        1.194194
BFGS:    4 18:12:45     -756.090459        0.805332
BFGS:    5 18:12:46     -756.724393        0.954973
BFGS:    6 18:12:47     -757.071578        1.035259
BFGS:    7 18:12:48     -757.676032        0.950695
BFGS:    8 18:12:49     -758.224723        0.603494
BFGS:    9 18:12:50     -758.613105        0.454934
BFGS:   10 18:12:51     -758.753848        0.627440
BFGS:   11 18:12:51     -758.970557        0.584438
BFGS:   12 18:12:52     -759.162671        0.387205
BFGS:   13 18:12:53     -759.255647        0.283791
BFGS:   14 18:12:54     -759.313347        0.265232
BFGS:   15 18:12:55     -759.382086        0.362962
BFGS:   16 18:12:56     -759.467271        0.337228
BFGS:   17 18:12:57     -759.548983        0.299609
BFGS:   18 18:

/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


... (493 outputs left out)

/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


      Step     Time          Energy          fmax
BFGS:    0 14:15:31    -1553.285590        2.434667
BFGS:    1 14:15:53    -1558.101123        2.130021
BFGS:    2 14:15:55    -1571.438254        3.118145
BFGS:    3 14:17:03    -1573.497867        1.188765
BFGS:    4 14:38:55    -1576.302490        1.024827
BFGS:    5 14:38:57    -1580.925683        1.876691
BFGS:    6 14:38:59    -1582.034946        1.042942
BFGS:    7 14:39:06    -1584.055064        0.727452
BFGS:    8 14:39:09    -1586.053936        0.818127
BFGS:    9 14:39:10    -1586.537329        0.926361
BFGS:   10 14:39:12    -1587.470326        1.005179
BFGS:   11 14:39:13    -1588.363955        0.669608
BFGS:   12 14:39:15    -1589.074880        0.791233
BFGS:   13 14:39:16    -1589.545767        0.660594
BFGS:   14 14:39:17    -1589.953660        0.679337
BFGS:   15 14:39:19    -1590.323890        0.603177
BFGS:   16 14:39:20    -1590.752307        0.718412
BFGS:   17 14:39:22    -1591.129253        0.689271
BFGS:   18 14:

## Ranking plot (only if `run_mliap_relaxation = True` and candidates were scored)

In [ ]:
if results:
    labels   = [r['folder'].replace(f'{general_folder}/', '') for r in results]
    energies = [r['adhesion_energy'] for r in results]

    plt.figure(figsize=(max(4, len(labels) * 0.5), 4))
    plt.plot(energies, 'o-')
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.ylabel(r'$E_{\text{adhesion}}$ (J/m$^2$)')
    plt.axhline(0, color='grey', linewidth=0.8)
    plt.tight_layout()
    plt.savefig(f'{general_folder}/adhesion_ranking.png', dpi=200)
    plt.show()


# Finite-temperature adhesion free energy (optional)

At temperature $T$ the same expression is used with the free energies at that temperature, $G(T) = U_0(T) + F_{vib}(T)$ per atom, from
NPT runs (`<folder>/md/<T>K`, one per temperature) of the **interface**, the **film** and the **substrate** of every candidate, the last two
being the film and substrate atoms alone in the interface cell (`reference/film`, `reference/substrate`, shared by the candidates of a match).
The runs start from the relaxed structures. $A$ is the mean area of the interface cell in its run. The table reports the contribution of every
term of $G$ (`d_U0_J_m2`, `d_F_vib_J_m2`) and the flags of the runs.

**The cell of the runs.** By default the cell is fully flexible. Then the film and the substrate on their own relax the strain that
the interface cell imposes on them, so their free energy no longer contains it, and **the strain energy of the film ends up in the adhesion
free energy** (the 0 K adhesion energy above does not have it, because it keeps the references in the interface cell). A flag reports a
reference whose area differs from the interface. `md_fix_cell` freezes any direction of the cell of *all three* runs (they must be run
alike): `'xy'` keeps the in-plane lattice (the stacking direction relaxes), `'all'` keeps every lattice vector (constant volume). A fixed
cell has no thermal expansion, and its free energy is that of that cell, not of zero pressure.

In [ ]:
# Finite temperature (optional)
finite_temperature = True
operating_T        = 300.0           # K, temperature of the free energies (it need not be an MD temperature, see iterate-slabs)
md_temperatures    = [300.0]         # K, temperatures of the NPT runs
run_md             = True           # True: run the pending NPT runs here with the ML-IAP (slow on a CPU; use a GPU), or run them with VASP

# NPT settings, the same for the interface, the film and the substrate of every candidate
md_min_length  = 15.0                # Angstroms, minimum length of a and b of the MD cell
md_n_steps     = 20000
md_timestep    = 1.0                 # fs
md_loginterval = 10                  # a frame every md_loginterval * md_timestep fs
md_ttime       = 50.0                # fs, thermostat time constant
md_ptime       = 500.0               # fs, barostat time constant
md_equil_ps    = 2.0                 # ps discarded at the start of every run (thermalisation) when it is analysed; stored in md.json
md_fix_cell    = None                # None: fully flexible; 'xy': fixed in-plane lattice; 'all': fixed cell (see the note above)

In [ ]:
from libraries import thermal

md_folders = []
if finite_temperature:
    md_settings = dict(min_length=md_min_length, n_steps=md_n_steps, timestep=md_timestep, loginterval=md_loginterval,
                       ttime=md_ttime, ptime=md_ptime, equil_ps=md_equil_ps, fix_cell=md_fix_cell)
    for candidate in candidates:
        for folder in (candidate['candidate_folder'], f"{candidate['reference_folder']}/film", f"{candidate['reference_folder']}/substrate"):
            if folder not in md_folders:
                md_folders.append(folder)
    for folder in md_folders:
        energy_lookup.ensure_relaxed(folder, model_load_path, relax_cell=False, device=device)     # -> <folder>/relaxation/CONTCAR
        thermal.prepare_md_folders(folder, 'slab', md_temperatures, start_from='relaxed', **md_settings)
    print(f'{len(md_folders)} structure(s) x {len(md_temperatures)} temperature(s); cell {md_fix_cell!r}')
else:
    print('finite_temperature = False: only the 0 K adhesion energies are computed.')

In [ ]:
if finite_temperature:
    if run_md:
        thermal.run_pending_md_in(md_folders, model_load_path=model_load_path, device=device)

    md_status = thermal.md_status_of([(f.replace(general_folder + '/', ''), f) for f in md_folders])
    print(md_status['state'].value_counts().to_string())
    if (md_status['state'] != 'done').any():
        print('\nCandidates with unfinished runs are left out (NaN) of the free-energy table:')
        display(md_status[md_status['state'] != 'done'])
        if not run_md:
            print('[note] they have not been run: set run_md = True to run them here, or run them with VASP.')

In [ ]:
adhesion_free_energy = None
if finite_temperature:
    thermal.check_md_consistency(md_folders, model_load_path)
    adhesion_free_energy = heterostructure.adhesion_free_energy_ranking(candidates, operating_T)
    display(adhesion_free_energy.drop(columns='flags'))
    print(f"Kendall tau between the 0 K and the {operating_T:g} K rankings: {adhesion_free_energy.attrs['kendall_tau']:.2f}")
    for folder, row in adhesion_free_energy[adhesion_free_energy['flags'] != ''].iterrows():
        print(f"\n[flags] {folder.replace(general_folder + '/', '')}: {row['flags']}")
    adhesion_free_energy.to_csv(f'{general_folder}/heterostructure_free_energy_{operating_T:g}K.csv')

In [ ]:
if adhesion_free_energy is not None and adhesion_free_energy['adhesion_T_J_m2'].notna().any():
    valid  = adhesion_free_energy.dropna(subset=['adhesion_T_J_m2'])
    labels = [f.replace(f'{general_folder}/', '') + ('*' if flags else '') for f, flags in zip(valid.index, valid['flags'])]

    plt.figure(figsize=(max(4, len(labels) * 0.5), 4))
    plt.plot(valid['adhesion_0K_J_m2'].values, 'o-', label='0 K')
    plt.plot(valid['adhesion_T_J_m2'].values, 's-', label=f'{operating_T:g} K')
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.ylabel(r'$\Delta G_{\text{adhesion}}$ (J/m$^2$)')
    plt.axhline(0, color='grey', linewidth=0.8)
    plt.title('sorted by the free energy at T (* has flags)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{general_folder}/adhesion_free_energy_{operating_T:g}K.png', dpi=200)
    plt.show()